# Phase 4: train Transolver on the SU2 hypersonic dataset

Kaggle orchestrator. Attach the `zeteixeira/su2-hypersonic-sphere-cone`
dataset as input, set Accelerator to GPU, Internet on, Run All.

Clones the repo, installs the missing dep, and runs
`scripts/phase4_train_su2.py`. Outputs (checkpoints, norm stats,
history, final eval JSON) land in `/kaggle/working/run/` and survive
the session as notebook output.

In [ ]:
import os, subprocess, sys, zipfile

REPO = "/kaggle/working/transolver-hypersonic"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/zeteixeira03/transolver-hypersonic.git", REPO],
                   check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops"], check=True)

# auto-detect the dataset mount: kaggle may name the input dir after either
# the dataset slug or the title, so score every mounted input instead of
# hard-coding a path
INPUT = "/kaggle/input"
mounts = sorted(os.listdir(INPUT)) if os.path.isdir(INPUT) else []
print("inputs mounted:", mounts)
assert mounts, "no input attached: right panel -> Input -> Add Input -> your su2 dataset"

def _score(d):
    p = os.path.join(INPUT, d)
    try:
        names = os.listdir(p)
    except OSError:
        return -1
    n_cases = sum(1 for f in names if f.startswith("case_"))
    has_zip = 1000 if "su2_cases.zip" in names else 0
    return n_cases + has_zip

DATA = os.path.join(INPUT, max(mounts, key=_score))
print("using:", DATA)
print("first entries:", sorted(os.listdir(DATA))[:5])

# the dataset arrives either extracted (case_*.npz + ledger.db at the root)
# or as the single su2_cases.zip if kaggle did not auto-extract the archive
if not os.path.isfile(f"{DATA}/ledger.db"):
    zsrc = f"{DATA}/su2_cases.zip"
    assert os.path.isfile(zsrc), (
        f"neither ledger.db nor su2_cases.zip under {DATA}; "
        f"contents: {sorted(os.listdir(DATA))[:10]}"
    )
    extracted = "/kaggle/working/su2_data"
    if not os.path.isfile(f"{extracted}/ledger.db"):
        os.makedirs(extracted, exist_ok=True)
        with zipfile.ZipFile(zsrc) as z:
            z.extractall(extracted)
    DATA = extracted

n_cases = len([d for d in os.listdir(DATA) if d.startswith("case_")])
print("cases:", n_cases)
assert n_cases > 100, f"only {n_cases} case files; wrong dataset attached?"

In [ ]:
!cd {REPO} && python scripts/phase4_train_su2.py \
    --workdir {DATA} \
    --out /kaggle/working/run \
    --epochs 200 --val-every 10 --seed 0

In [ ]:
import json
with open("/kaggle/working/run/final_eval.json") as f:
    final = json.load(f)
print(json.dumps(final["final"], indent=2))
print("splits:", final["splits"])